# IMDB Sentiment Analysis: RNN to DistilBERT

This notebook implements the upgrade path through phase 5 using the local IMDB Dataset.csv file.

- Phase 1: true word sequences, padding, and learned embeddings
- Phase 2: an experimental comparison of vanilla RNN, LSTM, and GRU
- Phase 3: bidirectional LSTM context modelling
- Phase 4: attention-based BiLSTM with token-level importance
- Phase 5: DistilBERT fine-tuning and a benchmark table

The earlier TF-IDF to RNN approach has been removed: a TF-IDF vector is not a meaningful time sequence for an RNN. TF-IDF remains only as a conventional logistic-regression baseline.

## Before you run this notebook

Use a Python environment with Python 3.10 or newer. Install the dependencies once with:

    pip install pandas numpy scikit-learn matplotlib seaborn torch transformers

For phase 5, the first run downloads distilbert-base-uncased from Hugging Face. A CUDA-enabled GPU is strongly recommended for full-dataset transformer training. Set FAST_RUN to True for a small smoke test; set it to False for your final experiment.

In [ ]:
from collections import Counter
import gc
import html
from pathlib import Path
import random
import re
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from torch.utils.data import DataLoader, Dataset

SEED = 42
FAST_RUN = False  # Set True to validate the notebook quickly on a smaller, balanced subset.

MAX_VOCAB_SIZE = 20_000
MAX_SEQUENCE_LENGTH = 250
BATCH_SIZE = 128
SEQUENCE_EPOCHS = 2 if FAST_RUN else 5
SEQUENCE_LEARNING_RATE = 1e-3

# These limits are only used when FAST_RUN is True. None means use every split row.
MAX_TRAIN_SAMPLES = 8_000 if FAST_RUN else None
MAX_VALIDATION_SAMPLES = 1_000 if FAST_RUN else None
MAX_TEST_SAMPLES = 1_000 if FAST_RUN else None


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')


## Load and clean the IMDB data

Cleaning deliberately preserves stopwords and word forms. Words such as not, never, and very carry sequential sentiment information; stemming and stopword removal would weaken an RNN, LSTM, GRU, or transformer.

In [ ]:
DATA_PATH = Path('IMDB Dataset.csv')
if not DATA_PATH.exists():
    raise FileNotFoundError(f'Expected the dataset at {DATA_PATH.resolve()}')


def clean_review(text):
    text = html.unescape(str(text))
    text = re.sub(r'<br\s*/?>', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip().lower()
    return text


df = pd.read_csv(DATA_PATH, usecols=['review', 'sentiment']).dropna().drop_duplicates().copy()
df['review'] = df['review'].map(clean_review)
df['label'] = df['sentiment'].map({'negative': 0, 'positive': 1})

if df['label'].isna().any():
    raise ValueError('The sentiment column must contain only positive and negative labels.')

df['label'] = df['label'].astype(int)
print(f'Rows after cleaning and de-duplication: {len(df):,}')
display(df[['review', 'sentiment']].head())
print(df['sentiment'].value_counts())


In [ ]:
def balanced_limit(frame, max_rows, seed=SEED):
    """Keep a reproducible balanced subset for smoke tests; None keeps all rows."""
    if max_rows is None or len(frame) <= max_rows:
        return frame.reset_index(drop=True)
    per_class = max_rows // frame['label'].nunique()
    sampled = frame.groupby('label', group_keys=False).apply(
        lambda group: group.sample(n=min(len(group), per_class), random_state=seed)
    )
    return sampled.sample(frac=1, random_state=seed).reset_index(drop=True)


# Stratification makes every split keep the positive/negative class balance.
train_df, holdout_df = train_test_split(
    df[['review', 'label']], test_size=0.20, stratify=df['label'], random_state=SEED
)
validation_df, test_df = train_test_split(
    holdout_df, test_size=0.50, stratify=holdout_df['label'], random_state=SEED
)

train_df = balanced_limit(train_df, MAX_TRAIN_SAMPLES)
validation_df = balanced_limit(validation_df, MAX_VALIDATION_SAMPLES)
test_df = balanced_limit(test_df, MAX_TEST_SAMPLES)

for split_name, split in [('train', train_df), ('validation', validation_df), ('test', test_df)]:
    print(f'{split_name:10s}: {len(split):,} rows | positive rate = {split.label.mean():.2%}')


## Phase 1: Tokenization, vocabulary, padding, and embeddings

The vocabulary is built from training reviews only, preventing information from validation or test reviews leaking into the model.

In [ ]:
PAD_TOKEN, UNK_TOKEN = '<PAD>', '<UNK>'
PAD_ID, UNK_ID = 0, 1
TOKEN_PATTERN = re.compile(r"[a-z]+(?:'[a-z]+)?")


def tokenize(text):
    return TOKEN_PATTERN.findall(text)


def build_vocabulary(texts, max_size=MAX_VOCAB_SIZE):
    counter = Counter()
    for text in texts:
        counter.update(tokenize(text))
    vocabulary = {PAD_TOKEN: PAD_ID, UNK_TOKEN: UNK_ID}
    for word, _ in counter.most_common(max_size - len(vocabulary)):
        vocabulary[word] = len(vocabulary)
    return vocabulary


vocab = build_vocabulary(train_df['review'])
print(f'Vocabulary size: {len(vocab):,}')


def encode_review(text, vocabulary=vocab, max_length=MAX_SEQUENCE_LENGTH):
    token_ids = [vocabulary.get(token, UNK_ID) for token in tokenize(text)[:max_length]]
    token_ids = token_ids or [UNK_ID]
    length = len(token_ids)
    token_ids += [PAD_ID] * (max_length - length)
    return token_ids, length


example_ids, example_length = encode_review(train_df.iloc[0].review)
print('Encoded shape:', len(example_ids), '| original sequence length:', example_length)


In [ ]:
class SentimentDataset(Dataset):
    def __init__(self, reviews, labels):
        self.features = [encode_review(review) for review in reviews]
        self.labels = np.asarray(labels, dtype=np.float32)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        token_ids, length = self.features[index]
        return (
            torch.tensor(token_ids, dtype=torch.long),
            torch.tensor(length, dtype=torch.long),
            torch.tensor(self.labels[index], dtype=torch.float32),
        )


pin_memory = device.type == 'cuda'
train_loader = DataLoader(SentimentDataset(train_df.review, train_df.label), batch_size=BATCH_SIZE, shuffle=True, pin_memory=pin_memory)
validation_loader = DataLoader(SentimentDataset(validation_df.review, validation_df.label), batch_size=BATCH_SIZE, shuffle=False, pin_memory=pin_memory)
test_loader = DataLoader(SentimentDataset(test_df.review, test_df.label), batch_size=BATCH_SIZE, shuffle=False, pin_memory=pin_memory)

batch_tokens, batch_lengths, batch_labels = next(iter(train_loader))
print('Token batch:', tuple(batch_tokens.shape), '| lengths:', tuple(batch_lengths.shape), '| labels:', tuple(batch_labels.shape))


## Shared sequence model and evaluation utilities

The packed sequence makes the recurrent layer ignore PAD tokens. Outputs are raw logits, so BCEWithLogitsLoss handles the sigmoid internally in a numerically stable way.

In [ ]:
class RecurrentClassifier(nn.Module):
    def __init__(self, vocabulary_size, cell_type='lstm', embedding_dim=128, hidden_size=128, num_layers=2, dropout=0.30, bidirectional=False):
        super().__init__()
        self.bidirectional = bidirectional
        self.num_layers = num_layers
        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(vocabulary_size, embedding_dim, padding_idx=PAD_ID)
        recurrent_layer = {'rnn': nn.RNN, 'lstm': nn.LSTM, 'gru': nn.GRU}[cell_type.lower()]
        self.recurrent = recurrent_layer(
            embedding_dim, hidden_size, num_layers=num_layers, batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0, bidirectional=bidirectional
        )
        output_features = hidden_size * (2 if bidirectional else 1)
        self.classifier = nn.Sequential(nn.Dropout(dropout), nn.Linear(output_features, 1))

    def forward(self, token_ids, lengths):
        embedded = self.embedding(token_ids)
        packed = pack_padded_sequence(embedded, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, hidden = self.recurrent(packed)
        if isinstance(hidden, tuple):  # LSTM returns (hidden_state, cell_state)
            hidden = hidden[0]
        if self.bidirectional:
            hidden = hidden.view(self.num_layers, 2, token_ids.size(0), self.hidden_size)
            features = torch.cat((hidden[-1, 0], hidden[-1, 1]), dim=1)
        else:
            features = hidden[-1]
        return self.classifier(features).squeeze(1)


def count_parameters(model):
    return sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)


def metrics_from_probabilities(labels, probabilities):
    labels = np.asarray(labels, dtype=int)
    predictions = (np.asarray(probabilities) >= 0.5).astype(int)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='binary', zero_division=0)
    return {
        'test_accuracy': float((predictions == labels).mean()),
        'test_precision': float(precision),
        'test_recall': float(recall),
        'test_f1': float(f1),
        'confusion_matrix': confusion_matrix(labels, predictions),
    }


@torch.no_grad()
def evaluate_sequence_model(model, loader):
    model.eval()
    all_labels, all_probabilities = [], []
    for token_ids, lengths, labels in loader:
        logits = model(token_ids.to(device), lengths)
        all_probabilities.extend(torch.sigmoid(logits).cpu().numpy())
        all_labels.extend(labels.numpy())
    return metrics_from_probabilities(all_labels, all_probabilities)


def train_sequence_model(model, epochs=SEQUENCE_EPOCHS, learning_rate=SEQUENCE_LEARNING_RATE):
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=1e-4)
    criterion = nn.BCEWithLogitsLoss()
    best_state, best_validation_f1 = None, -np.inf
    history = []

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        for token_ids, lengths, labels in train_loader:
            optimizer.zero_grad(set_to_none=True)
            logits = model(token_ids.to(device), lengths)
            loss = criterion(logits, labels.to(device))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            running_loss += loss.item() * labels.size(0)

        validation_metrics = evaluate_sequence_model(model, validation_loader)
        epoch_loss = running_loss / len(train_loader.dataset)
        history.append({'epoch': epoch, 'train_loss': epoch_loss, 'validation_f1': validation_metrics['test_f1']})
        print(f'Epoch {epoch}/{epochs} | train loss: {epoch_loss:.4f} | validation F1: {validation_metrics["test_f1"]:.4f}')
        if validation_metrics['test_f1'] > best_validation_f1:
            best_validation_f1 = validation_metrics['test_f1']
            best_state = {name: value.detach().cpu().clone() for name, value in model.state_dict().items()}

    model.load_state_dict(best_state)
    return pd.DataFrame(history)


def fit_sequence_experiment(model_name, model):
    started_at = time.perf_counter()
    history = train_sequence_model(model)
    elapsed_seconds = time.perf_counter() - started_at
    test_metrics = evaluate_sequence_model(model, test_loader)
    result = {
        'model': model_name,
        'parameters': count_parameters(model),
        'train_seconds': elapsed_seconds,
        **{key: value for key, value in test_metrics.items() if key != 'confusion_matrix'},
    }
    return history, test_metrics, result


## Conventional baseline: TF-IDF plus logistic regression

This is not fed into an RNN. It provides a fair traditional NLP reference point for the phase-5 benchmark.

In [ ]:
baseline_started_at = time.perf_counter()
tfidf = TfidfVectorizer(max_features=20_000, ngram_range=(1, 2), min_df=2, sublinear_tf=True)
X_train_tfidf = tfidf.fit_transform(train_df.review)
X_test_tfidf = tfidf.transform(test_df.review)

baseline_model = LogisticRegression(max_iter=1_000, random_state=SEED)
baseline_model.fit(X_train_tfidf, train_df.label)
baseline_probabilities = baseline_model.predict_proba(X_test_tfidf)[:, 1]
baseline_metrics = metrics_from_probabilities(test_df.label, baseline_probabilities)

results = [{
    'model': 'TF-IDF + Logistic Regression',
    'parameters': X_train_tfidf.shape[1],  # feature count, not neural-network weights
    'train_seconds': time.perf_counter() - baseline_started_at,
    **{key: value for key, value in baseline_metrics.items() if key != 'confusion_matrix'},
}]
print(pd.DataFrame(results).drop(columns='parameters'))


## Phase 2: Compare vanilla RNN, LSTM, and GRU

All models use the same vocabulary, split, batch size, hidden size, and training procedure, making the comparison meaningful.

In [ ]:
sequence_histories = {}
sequence_test_metrics = {}

for display_name, cell_type in [('Vanilla RNN', 'rnn'), ('LSTM', 'lstm'), ('GRU', 'gru')]:
    print(f'\nTraining {display_name}')
    set_seed()
    model = RecurrentClassifier(len(vocab), cell_type=cell_type).to(device)
    history, test_metrics, result = fit_sequence_experiment(display_name, model)
    sequence_histories[display_name] = history
    sequence_test_metrics[display_name] = test_metrics
    results.append(result)
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

pd.DataFrame(results).drop(columns='parameters')


## Phase 3: Bidirectional LSTM

A BiLSTM reads every review from left to right and right to left. Its final representation includes context from both directions.

In [ ]:
print('Training BiLSTM')
set_seed()
bilstm_model = RecurrentClassifier(len(vocab), cell_type='lstm', bidirectional=True).to(device)
bilstm_history, bilstm_test_metrics, bilstm_result = fit_sequence_experiment('BiLSTM', bilstm_model)
sequence_histories['BiLSTM'] = bilstm_history
sequence_test_metrics['BiLSTM'] = bilstm_test_metrics
results.append(bilstm_result)

del bilstm_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


## Phase 4: Attention-based BiLSTM

Attention learns a weight for each non-padding token. The attention plot is an interpretability aid, not a proof of causality.

In [ ]:
class AdditiveAttention(nn.Module):
    def __init__(self, feature_size, attention_size):
        super().__init__()
        self.projection = nn.Linear(feature_size, attention_size)
        self.score = nn.Linear(attention_size, 1, bias=False)

    def forward(self, sequence_outputs, mask):
        scores = self.score(torch.tanh(self.projection(sequence_outputs))).squeeze(-1)
        scores = scores.masked_fill(~mask, -1e9)
        weights = torch.softmax(scores, dim=1)
        context = torch.bmm(weights.unsqueeze(1), sequence_outputs).squeeze(1)
        return context, weights


class BiLSTMAttentionClassifier(nn.Module):
    def __init__(self, vocabulary_size, embedding_dim=128, hidden_size=128, num_layers=2, dropout=0.30):
        super().__init__()
        self.embedding = nn.Embedding(vocabulary_size, embedding_dim, padding_idx=PAD_ID)
        self.lstm = nn.LSTM(
            embedding_dim, hidden_size, num_layers=num_layers, batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0, bidirectional=True
        )
        self.attention = AdditiveAttention(hidden_size * 2, hidden_size)
        self.classifier = nn.Sequential(nn.Dropout(dropout), nn.Linear(hidden_size * 2, 1))

    def forward(self, token_ids, lengths, return_attention=False):
        embedded = self.embedding(token_ids)
        packed = pack_padded_sequence(embedded, lengths.cpu(), batch_first=True, enforce_sorted=False)
        packed_outputs, _ = self.lstm(packed)
        outputs, _ = pad_packed_sequence(packed_outputs, batch_first=True, total_length=token_ids.size(1))
        positions = torch.arange(token_ids.size(1), device=token_ids.device).unsqueeze(0)
        mask = positions < lengths.to(token_ids.device).unsqueeze(1)
        context, weights = self.attention(outputs, mask)
        logits = self.classifier(context).squeeze(1)
        return (logits, weights) if return_attention else logits


print('Training BiLSTM + Attention')
set_seed()
attention_model = BiLSTMAttentionClassifier(len(vocab)).to(device)
attention_history, attention_test_metrics, attention_result = fit_sequence_experiment('BiLSTM + Attention', attention_model)
sequence_histories['BiLSTM + Attention'] = attention_history
sequence_test_metrics['BiLSTM + Attention'] = attention_test_metrics
results.append(attention_result)


In [ ]:
def plot_attention(review, model=attention_model, top_k=20):
    cleaned = clean_review(review)
    token_ids, length = encode_review(cleaned)
    tokens = tokenize(cleaned)[:length]
    inputs = torch.tensor([token_ids], dtype=torch.long, device=device)
    lengths = torch.tensor([length], dtype=torch.long)

    model.eval()
    with torch.no_grad():
        logits, weights = model(inputs, lengths, return_attention=True)
        probability = torch.sigmoid(logits).item()
    weights = weights[0, :length].cpu().numpy()
    top_indices = np.argsort(weights)[-top_k:]

    plt.figure(figsize=(10, max(4, top_k * 0.28)))
    plt.barh([tokens[index] for index in top_indices], weights[top_indices], color='#4C78A8')
    plt.xlabel('Attention weight')
    prediction = 'positive' if probability >= 0.5 else 'negative'
    plt.title(f'Prediction: {prediction} ({probability:.1%} positive probability)')
    plt.tight_layout()
    plt.show()


plot_attention('The acting was phenomenal, but the story was painfully slow and the ending was disappointing.')


## Phase 5: Fine-tune DistilBERT

This branch keeps the same train, validation, and test split so its reported test metrics can be compared with the traditional and recurrent approaches. It requires the transformers package and an internet connection on the first run to download the pretrained weights.

In [ ]:
try:
    from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup
except ImportError as error:
    raise ImportError('Install phase-5 dependencies with: pip install transformers') from error

TRANSFORMER_MODEL_NAME = 'distilbert-base-uncased'
TRANSFORMER_MAX_LENGTH = 256
TRANSFORMER_EPOCHS = 1 if FAST_RUN else 3
TRANSFORMER_BATCH_SIZE = 16 if device.type == 'cuda' else 4
TRANSFORMER_LEARNING_RATE = 2e-5
TRANSFORMER_TRAIN_LIMIT = 4_000 if FAST_RUN else None
TRANSFORMER_VALIDATION_LIMIT = 500 if FAST_RUN else None
TRANSFORMER_TEST_LIMIT = 500 if FAST_RUN else None


class TransformerReviewDataset(Dataset):
    def __init__(self, reviews, labels, tokenizer):
        self.encodings = tokenizer(
            list(reviews), truncation=True, padding='max_length', max_length=TRANSFORMER_MAX_LENGTH
        )
        self.labels = list(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        item = {key: torch.tensor(value[index], dtype=torch.long) for key, value in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[index], dtype=torch.long)
        return item


@torch.no_grad()
def evaluate_transformer(model, loader):
    model.eval()
    all_labels, all_probabilities = [], []
    for batch in loader:
        labels = batch.pop('labels')
        batch = {name: value.to(device) for name, value in batch.items()}
        logits = model(**batch).logits
        all_probabilities.extend(torch.softmax(logits, dim=1)[:, 1].cpu().numpy())
        all_labels.extend(labels.numpy())
    return metrics_from_probabilities(all_labels, all_probabilities)


def train_transformer(model, train_loader, validation_loader):
    optimizer = torch.optim.AdamW(model.parameters(), lr=TRANSFORMER_LEARNING_RATE, weight_decay=0.01)
    total_steps = len(train_loader) * TRANSFORMER_EPOCHS
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps)
    best_state, best_validation_f1 = None, -np.inf
    history = []

    for epoch in range(1, TRANSFORMER_EPOCHS + 1):
        model.train()
        running_loss = 0.0
        for batch in train_loader:
            labels = batch.pop('labels').to(device)
            batch = {name: value.to(device) for name, value in batch.items()}
            optimizer.zero_grad(set_to_none=True)
            output = model(**batch, labels=labels)
            output.loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            running_loss += output.loss.item() * labels.size(0)

        validation_metrics = evaluate_transformer(model, validation_loader)
        epoch_loss = running_loss / len(train_loader.dataset)
        history.append({'epoch': epoch, 'train_loss': epoch_loss, 'validation_f1': validation_metrics['test_f1']})
        print(f'Epoch {epoch}/{TRANSFORMER_EPOCHS} | train loss: {epoch_loss:.4f} | validation F1: {validation_metrics["test_f1"]:.4f}')
        if validation_metrics['test_f1'] > best_validation_f1:
            best_validation_f1 = validation_metrics['test_f1']
            best_state = {name: value.detach().cpu().clone() for name, value in model.state_dict().items()}

    model.load_state_dict(best_state)
    return pd.DataFrame(history)


In [ ]:
# This cell downloads the pretrained model the first time it is run.
transformer_train_df = balanced_limit(train_df, TRANSFORMER_TRAIN_LIMIT)
transformer_validation_df = balanced_limit(validation_df, TRANSFORMER_VALIDATION_LIMIT)
transformer_test_df = balanced_limit(test_df, TRANSFORMER_TEST_LIMIT)

set_seed()
transformer_tokenizer = AutoTokenizer.from_pretrained(TRANSFORMER_MODEL_NAME)
transformer_train_loader = DataLoader(
    TransformerReviewDataset(transformer_train_df.review, transformer_train_df.label, transformer_tokenizer),
    batch_size=TRANSFORMER_BATCH_SIZE, shuffle=True, pin_memory=pin_memory
)
transformer_validation_loader = DataLoader(
    TransformerReviewDataset(transformer_validation_df.review, transformer_validation_df.label, transformer_tokenizer),
    batch_size=TRANSFORMER_BATCH_SIZE, shuffle=False, pin_memory=pin_memory
)
transformer_test_loader = DataLoader(
    TransformerReviewDataset(transformer_test_df.review, transformer_test_df.label, transformer_tokenizer),
    batch_size=TRANSFORMER_BATCH_SIZE, shuffle=False, pin_memory=pin_memory
)

transformer_started_at = time.perf_counter()
transformer_model = AutoModelForSequenceClassification.from_pretrained(TRANSFORMER_MODEL_NAME, num_labels=2).to(device)
transformer_history = train_transformer(transformer_model, transformer_train_loader, transformer_validation_loader)
transformer_test_metrics = evaluate_transformer(transformer_model, transformer_test_loader)

results.append({
    'model': 'DistilBERT (fine-tuned)',
    'parameters': count_parameters(transformer_model),
    'train_seconds': time.perf_counter() - transformer_started_at,
    **{key: value for key, value in transformer_test_metrics.items() if key != 'confusion_matrix'},
})


## Final benchmark and diagnostics

Only compare values from runs using the same data limits. With FAST_RUN set to False, every model uses the complete held-out test set.

In [ ]:
results_df = pd.DataFrame(results).sort_values('test_f1', ascending=False).reset_index(drop=True)
metric_columns = ['model', 'test_accuracy', 'test_precision', 'test_recall', 'test_f1', 'parameters', 'train_seconds']
display(results_df[metric_columns].style.format({
    'test_accuracy': '{:.2%}', 'test_precision': '{:.2%}', 'test_recall': '{:.2%}',
    'test_f1': '{:.2%}', 'parameters': '{:,.0f}', 'train_seconds': '{:.1f}'
}))

plt.figure(figsize=(10, 5))
sns.barplot(data=results_df, x='test_f1', y='model', palette='viridis')
plt.xlim(0, 1)
plt.xlabel('Held-out test F1 score')
plt.ylabel('')
plt.title('IMDB sentiment model benchmark')
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
sns.heatmap(attention_test_metrics['confusion_matrix'], annot=True, fmt='d', cbar=False, cmap='Blues', ax=axes[0])
axes[0].set(title='BiLSTM + Attention', xlabel='Predicted label', ylabel='True label')
sns.heatmap(transformer_test_metrics['confusion_matrix'], annot=True, fmt='d', cbar=False, cmap='Greens', ax=axes[1])
axes[1].set(title='Fine-tuned DistilBERT', xlabel='Predicted label', ylabel='True label')
plt.tight_layout()
plt.show()


## What each completed phase demonstrates

| Phase | Project capability | Resume-ready phrase |
|---|---|---|
| 1 | Token sequences, padding, and embeddings | Developed a sequence-based sentiment classifier using learned word embeddings and recurrent neural networks. |
| 2 | Controlled RNN/LSTM/GRU comparison | Conducted a comparative analysis of RNN, LSTM, and GRU architectures for sentiment classification. |
| 3 | Bidirectional context | Applied bidirectional sequence modelling to capture contextual sentiment information. |
| 4 | Attention plus token weights | Built attention-based sentiment classification with interpretable token-level importance visualizations. |
| 5 | Pretrained Transformer comparison | Benchmarked traditional, recurrent, and transformer-based approaches for IMDB sentiment analysis. |

For the final report, record the displayed test metrics, training time, hardware used, data limits, and the random seed. Do not claim the best model until you have run the full-data configuration and saved its outputs.